# Lab 2 — From Pandas to Spark: the DataFrame API

# Apache Spark (Local) vs Pandas: *From Single-Machine to Distributed Thinking*



---
## 1. Setup & Architecture

### How Spark works

Spark distributes data into **partitions** that are processed in parallel across multiple nodes.  
In **local mode**, all of this happens on your single machine — perfect for learning.

```
┌──────────────────────────────────────────────────────────────┐
│                       SPARK CLUSTER                          │
│                                                              │
│   ┌───────────┐     ┌────────────┐     ┌────────────┐        │
│   │  Driver   │────▶│  Executor  │     │  Executor  │        │
│   │ (your code│     │ (Worker 1) │     │ (Worker 2) │        │
│   │  & plans) │     │ Partition1 │     │ Partition2 │        │
│   └───────────┘     └────────────┘     └────────────┘        │
│                                                              │
│         local[*] mode → everything runs on ONE machine       │
└──────────────────────────────────────────────────────────────┘
```


### Pandas vs Spark at a glance

| | Pandas | Spark |
|-|--------|-------|
| **Runs on** | 1 machine, 1 process | Cluster of machines (or local) |
| **Data lives** | In RAM | Distributed across partitions |
| **Execution** | Immediate (eager) | Lazy — runs only on an action |
| **Scale** | Up to a few GBs | Terabytes / Petabytes |
| **Entry point** | `import pandas as pd` | `SparkSession` |

In [ ]:
# Install if needed
# !pip install pyspark

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F   # Spark's built-in function library
from pyspark.sql.types import *          # schema type definitions
from pyspark.sql.window import Window    # window function specs

import pyspark
print(f"Pandas  : {pd.__version__}")
print(f"PySpark : {pyspark.__version__}")

Pandas  : 2.2.3
PySpark : 4.0.4


In [ ]:
# Creating the SparkSession — the single entry point for all Spark functionality.
#   local[*]  → use ALL available CPU cores
#   local[2]  → use exactly 2 cores

spark = (
    SparkSession.builder
    .appName("Spark vs Pandas - Class")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")         # small dataset, fewer partitions
    .config("spark.sql.repl.eagerEval.enabled", True)    # pretty display in notebooks
    .getOrCreate()
)

spark

In [ ]:
# While cells are running, open the Spark UI in your browser
print(f"Spark UI : http://localhost:4040")
print(f"Version  : {spark.version}")
print(f"App Name : {spark.sparkContext.appName}")
print(f"Master   : {spark.sparkContext.master}")

Spark UI : http://localhost:4040
Version  : 4.0.4
App Name : Spark vs Pandas - Class
Master   : local[*]


---
## 2. Creating DataFrames

> We'll use an e-commerce sales dataset throughout the entire notebook.

In [ ]:
# Shared raw data used by both Pandas and Spark
data = {
    "order_id":  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
                  1011, 1012, 1013, 1014, 1015],
    "product":   ["Notebook", "Mouse", "Keyboard", "Monitor", "Headset",
                  "Webcam", "External HD", "Pendrive", "USB Cable", "USB Hub",
                  "Notebook", "Monitor", "Keyboard", "Mouse", "Headset"],
    "category":  ["Computers", "Peripherals", "Peripherals", "Computers", "Audio",
                  "Peripherals", "Storage", "Storage", "Cables", "Peripherals",
                  "Computers", "Computers", "Peripherals", "Peripherals", "Audio"],
    "price":     [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120,
                  3200, 1100, 145, 90, 280],
    "quantity":  [1, 3, 2, 1, 2, 1, 1, 5, 10, 2,
                  2, 1, 3, 2, 1],
    "rating":    [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3,
                  4.9, 4.6, 4.4, 4.1, 4.0],
    "region":    ["South", "North", "Southeast", "Southeast", "North",
                  "South", "Southeast", "North", "South", "Southeast",
                  "North", "South", "Southeast", "North", "Southeast"],
    "sale_date": ["2024-01-15", "2024-01-18", "2024-02-03", "2024-02-10", "2024-02-14",
                  "2024-03-01", "2024-03-15", "2024-03-22", "2024-04-05", "2024-04-18",
                  "2024-05-02", "2024-05-20", "2024-06-08", "2024-06-15", "2024-06-30"]
}

### 2.1 From a Python dictionary

In [ ]:
# ════════════════════════════════
#  PANDAS
# ════════════════════════════════
pdf = pd.DataFrame(data)
pdf.head(5)

,order_id,product,category,price,quantity,rating,region,sale_date
0,1001,Notebook,Computers,3500,1,4.8,South,2024-01-15
1,1002,Mouse,Peripherals,85,3,4.2,North,2024-01-18
2,1003,Keyboard,Peripherals,150,2,4.5,Southeast,2024-02-03
3,1004,Monitor,Computers,1200,1,4.7,Southeast,2024-02-10
4,1005,Headset,Audio,250,2,3.9,North,2024-02-14


In [ ]:
# ════════════════════════════════
#  SPARK
# ════════════════════════════════
# Most common in dev/learning: convert from a Pandas DataFrame
sdf = spark.createDataFrame(pdf)
sdf.show(5, truncate=False)

+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
|1004    |Monitor |Computers  |1200 |1       |4.7   |Southeast|2024-02-10|
|1005    |Headset |Audio      |250  |2       |3.9   |North    |2024-02-14|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 5 rows


In [ ]:
# With an explicit schema — recommended in production.
# Avoids schema inference costs and makes intent clear.

schema = StructType([
    StructField("order_id",  IntegerType(), nullable=False),
    StructField("product",   StringType(),  nullable=False),
    StructField("category",  StringType(),  nullable=True),
    StructField("price",     IntegerType(), nullable=False),
    StructField("quantity",  IntegerType(), nullable=False),
    StructField("rating",    DoubleType(),  nullable=True),
    StructField("region",    StringType(),  nullable=True),
    StructField("sale_date", StringType(),  nullable=True),
])

rows = list(zip(*[data[k] for k in schema.fieldNames()]))
sdf_typed = spark.createDataFrame(rows, schema=schema) # criando a tabela virtual
sdf_typed.printSchema()

root
 |-- order_id: integer (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- rating: double (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)



### 2.2 Reading files

In [ ]:
# Pandas readers
# pdf = pd.read_csv("sales.csv")
# pdf = pd.read_parquet("sales.parquet")
# pdf = pd.read_json("sales.json")

# vector: list[list] = [[1,2,3,4], [0.7,4,5,3,3]]
# read_csv, "string", ast.literal_eval

# Spark readers
# sdf = spark.read.csv("sales.csv", header=True, inferSchema=True)
# sdf = spark.read.parquet("sales.parquet")     # ← preferred columnar format
# sdf = spark.read.json("sales.json")
# sdf = spark.read.format("delta").load("path") # ← Delta Lake

# Round-trip through Parquet
pdf.to_parquet("/tmp/sales.parquet", index=False)
sdf_parquet = spark.read.parquet("/tmp/sales.parquet")

print("Read from Parquet:")
sdf_parquet.show(3, truncate=False)

Read from Parquet:
+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 3 rows


### 2.3 Converting between Pandas and Spark

> ⚠️ **`toPandas()`** collects **all data** to the driver node.  
> Only use this on small DataFrames — it defeats the purpose of distributed processing on large data.

In [ ]:
# Spark → Pandas  (collects everything to the driver)
pdf_from_spark = sdf.toPandas()
print("Spark → Pandas:", type(pdf_from_spark))

# Pandas → Spark
sdf_from_pandas = spark.createDataFrame(pdf)
print("Pandas → Spark:", type(sdf_from_pandas))

Spark → Pandas: <class 'pandas.core.frame.DataFrame'>
Pandas → Spark: <class 'pyspark.sql.classic.dataframe.DataFrame'>


In [ ]:
pdf["product"].value_counts()

,count
product,
Notebook,2
Mouse,2
Keyboard,2
Monitor,2
Headset,2
Webcam,1
External HD,1
Pendrive,1
USB Cable,1


---
## 3. Exploration

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Dimensions | `df.shape` | `df.count(), len(df.columns)` |
| First rows | `df.head(n)` | `df.show(n)` |
| Column types | `df.dtypes` | `df.printSchema()` |
| Statistics | `df.describe()` | `df.describe()` |
| Null counts | `df.isnull().sum()` | `df.select([F.count(F.when(F.isnull(c), c))...])` |
| Unique values | `df["col"].unique()` | `df.select("col").distinct()` |
| Value counts | `df["col"].value_counts()` | `df.groupBy("col").count()` |

In [ ]:
# Shape / Dimensions
print(f"🐼 Pandas  shape : {pdf.shape}")

# ⚠️ .count() triggers execution in Spark
# print(f"🔥 Spark   rows  : {sdf.count()}  |  cols: {len(sdf.columns)}")

🐼 Pandas  shape : (15, 8)


In [ ]:
# Schema / Types
print("🐼 Pandas dtypes:")
print(pdf.dtypes)
print()
print("🔥 Spark schema:")
sdf.printSchema()

🐼 Pandas dtypes:
order_id       int64
product       object
category      object
price          int64
quantity       int64
rating       float64
region        object
sale_date     object
dtype: object

🔥 Spark schema:
root
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)



In [ ]:
# First rows
print("🐼 Pandas .head():")
display(pdf.head(4))

print("\n🔥 Spark .show():")
sdf.show(4, truncate=False)

🐼 Pandas .head():


,order_id,product,category,price,quantity,rating,region,sale_date
0,1001,Notebook,Computers,3500,1,4.8,South,2024-01-15
1,1002,Mouse,Peripherals,85,3,4.2,North,2024-01-18
2,1003,Keyboard,Peripherals,150,2,4.5,Southeast,2024-02-03
3,1004,Monitor,Computers,1200,1,4.7,Southeast,2024-02-10



🔥 Spark .show():
+--------+--------+-----------+-----+--------+------+---------+----------+
|order_id|product |category   |price|quantity|rating|region   |sale_date |
+--------+--------+-----------+-----+--------+------+---------+----------+
|1001    |Notebook|Computers  |3500 |1       |4.8   |South    |2024-01-15|
|1002    |Mouse   |Peripherals|85   |3       |4.2   |North    |2024-01-18|
|1003    |Keyboard|Peripherals|150  |2       |4.5   |Southeast|2024-02-03|
|1004    |Monitor |Computers  |1200 |1       |4.7   |Southeast|2024-02-10|
+--------+--------+-----------+-----+--------+------+---------+----------+
only showing top 4 rows


In [ ]:
# Descriptive statistics
print("🐼 Pandas .describe():")
display(pdf.describe())

print("\n🔥 Spark .describe():")
sdf.describe("price", "quantity", "rating").show()

🐼 Pandas .describe():


,order_id,price,quantity,rating
count,15.000000,15.000000,15.000000,14.000000
mean,1008.000000,728.333333,2.466667,4.364286
std,4.472136,1123.252081,2.356349,0.324884
min,1001.000000,25.000000,1.000000,3.900000
25%,1004.500000,105.000000,1.000000,4.100000
50%,1008.000000,250.000000,2.000000,4.350000
75%,1011.500000,750.000000,2.500000,4.600000
max,1015.000000,3500.000000,10.000000,4.900000



🔥 Spark .describe():
+-------+------------------+-----------------+------+
|summary|             price|         quantity|rating|
+-------+------------------+-----------------+------+
|  count|                15|               15|    15|
|   mean| 728.3333333333334|2.466666666666667|   NaN|
| stddev|1123.2520812779464|2.356349072692956|   NaN|
|    min|                25|                1|   3.9|
|    max|              3500|               10|   NaN|
+-------+------------------+-----------------+------+



In [ ]:
# Null counts per column
print("🐼 Pandas isnull().sum():")
print(pdf.isnull().sum())

print("\n🔥 Spark null counts:")
sdf.select([
    F.count(F.when(F.isnull(c), c)).alias(c)
    for c in sdf.columns
]).show()

🐼 Pandas isnull().sum():
order_id     0
product      0
category     0
price        0
quantity     0
rating       1
region       0
sale_date    0
dtype: int64

🔥 Spark null counts:
+--------+-------+--------+-----+--------+------+------+---------+
|order_id|product|category|price|quantity|rating|region|sale_date|
+--------+-------+--------+-----+--------+------+------+---------+
|       0|      0|       0|    0|       0|     0|     0|        0|
+--------+-------+--------+-----+--------+------+------+---------+



In [ ]:
# Unique values and value counts
print("🐼 Pandas unique():")
print(pdf["category"].unique())
print()
print("🐼 Pandas value_counts():")
print(pdf["category"].value_counts())

print("\n🔥 Spark .distinct():")
sdf.select("category").distinct().show()

print("\n🔥 Spark groupBy().count() — equivalent to value_counts():")
sdf.groupBy("category").count().orderBy("count", ascending=False).show()

🐼 Pandas unique():
['Computers' 'Peripherals' 'Audio' 'Storage' 'Cables']

🐼 Pandas value_counts():
category
Peripherals    6
Computers      4
Audio          2
Storage        2
Cables         1
Name: count, dtype: int64

🔥 Spark .distinct():
+-----------+
|   category|
+-----------+
|  Computers|
|Peripherals|
|      Audio|
|    Storage|
|     Cables|
+-----------+


🔥 Spark groupBy().count() — equivalent to value_counts():
+-----------+-----+
|   category|count|
+-----------+-----+
|Peripherals|    6|
|  Computers|    4|
|      Audio|    2|
|    Storage|    2|
|     Cables|    1|
+-----------+-----+



---
## 4. Selection & Filtering

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Select columns | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| Filter rows | `df[df["col"] > x]` | `df.filter(df.col > x)` |
| SQL-style filter | `df.query("col > x")` | `df.filter("col > x")` |
| IN operator | `df[df["col"].isin([...])]` | `df.filter(df.col.isin([...]))` |
| Null filter | `df[df["col"].isnull()]` | `df.filter(F.col("col").isNull())` |
| Sort | `df.sort_values("col")` | `df.orderBy("col")` |

In [ ]:
# Selecting columns
print("🐼 Pandas:")
display(pdf[["order_id", "product", "price", "quantity"]].head(5))

print("\n🔥 Spark:")
sdf.select("order_id", "product", "price", "quantity").show(5)

🐼 Pandas:


,order_id,product,price,quantity
0,1001,Notebook,3500,1
1,1002,Mouse,85,3
2,1003,Keyboard,150,2
3,1004,Monitor,1200,1
4,1005,Headset,250,2



🔥 Spark:
+--------+--------+-----+--------+
|order_id| product|price|quantity|
+--------+--------+-----+--------+
|    1001|Notebook| 3500|       1|
|    1002|   Mouse|   85|       3|
|    1003|Keyboard|  150|       2|
|    1004| Monitor| 1200|       1|
|    1005| Headset|  250|       2|
+--------+--------+-----+--------+
only showing top 5 rows


In [ ]:
# Simple filter — Spark has two equivalent syntaxes
print("🐼 Pandas — price > 500:")
display(pdf[pdf["price"] > 500][["product", "price", "category"]])

print("\n🔥 Spark — column syntax:")
sdf.filter(sdf.price > 500).select("product", "price", "category").show()

print("\n🔥 Spark — SQL string syntax (mirrors Pandas .query()):")
sdf.filter("price > 500").select("product", "price", "category").show()

🐼 Pandas — price > 500:


,product,price,category
0,Notebook,3500,Computers
3,Monitor,1200,Computers
10,Notebook,3200,Computers
11,Monitor,1100,Computers



🔥 Spark — column syntax:
+--------+-----+---------+
| product|price| category|
+--------+-----+---------+
|Notebook| 3500|Computers|
| Monitor| 1200|Computers|
|Notebook| 3200|Computers|
| Monitor| 1100|Computers|
+--------+-----+---------+


🔥 Spark — SQL string syntax (mirrors Pandas .query()):
+--------+-----+---------+
| product|price| category|
+--------+-----+---------+
|Notebook| 3500|Computers|
| Monitor| 1200|Computers|
|Notebook| 3200|Computers|
| Monitor| 1100|Computers|
+--------+-----+---------+



In [ ]:
# Multiple conditions:  & = AND   | = OR   ~ = NOT
print("🐼 Pandas — price > 100 AND rating >= 4.5:")
mask = (pdf["price"] > 100) & (pdf["rating"] >= 4.5)
display(pdf[mask][["product", "price", "rating", "category"]])

print("\n🔥 Spark — same logic:")
sdf.filter(
    (F.col("price") > 100) & (F.col("rating") >= 4.5)
).select("product", "price", "rating", "category").show()

🐼 Pandas — price > 100 AND rating >= 4.5:


,product,price,rating,category
0,Notebook,3500,4.8,Computers
2,Keyboard,150,4.5,Peripherals
3,Monitor,1200,4.7,Computers
6,External HD,400,4.6,Storage
10,Notebook,3200,4.9,Computers
11,Monitor,1100,4.6,Computers



🔥 Spark — same logic:
+-----------+-----+------+-----------+
|    product|price|rating|   category|
+-----------+-----+------+-----------+
|   Notebook| 3500|   4.8|  Computers|
|   Keyboard|  150|   4.5|Peripherals|
|    Monitor| 1200|   4.7|  Computers|
|External HD|  400|   4.6|    Storage|
|   Notebook| 3200|   4.9|  Computers|
|    Monitor| 1100|   4.6|  Computers|
+-----------+-----+------+-----------+



In [ ]:
# isin — IN operator
targets = ["Audio", "Storage"]

# (pdf["category"] == "Audio") | (pdf["category"] == "Storage")

print("🐼 Pandas .isin():")
display(pdf[pdf["category"].isin(targets)][["product", "category", "price"]])

print("\n🔥 Spark .isin():")
sdf.filter(F.col("category").isin(targets)).select("product", "category", "price").show()

🐼 Pandas .isin():


,product,category,price
4,Headset,Audio,250
6,External HD,Storage,400
7,Pendrive,Storage,60
14,Headset,Audio,280



🔥 Spark .isin():
+-----------+--------+-----+
|    product|category|price|
+-----------+--------+-----+
|    Headset|   Audio|  250|
|External HD| Storage|  400|
|   Pendrive| Storage|   60|
|    Headset|   Audio|  280|
+-----------+--------+-----+



In [ ]:
# Filtering nulls
print("🐼 Pandas — rows where rating is null:")
display(pdf[pdf["rating"].isnull()])

print("\n🔥 Spark — rows where rating is null:")
sdf.filter(F.col("rating").isNull()).show()

🐼 Pandas — rows where rating is null:


,order_id,product,category,price,quantity,rating,region,sale_date
8,1009,USB Cable,Cables,25,10,NaN,South,2024-04-05



🔥 Spark — rows where rating is null:
+--------+-------+--------+-----+--------+------+------+---------+
|order_id|product|category|price|quantity|rating|region|sale_date|
+--------+-------+--------+-----+--------+------+------+---------+
+--------+-------+--------+-----+--------+------+------+---------+



In [ ]:
# Sorting by multiple columns
print("🐼 Pandas .sort_values():")
display(
    pdf.sort_values(["price", "rating"], ascending=[False, False])
    [["product", "price", "rating"]].head(5)
)

print("\n🔥 Spark .orderBy():")
sdf.orderBy(
    F.col("price").desc(), F.col("rating").desc()
).select("product", "price", "rating").show(5)

🐼 Pandas .sort_values():


,product,price,rating
0,Notebook,3500,4.8
10,Notebook,3200,4.9
3,Monitor,1200,4.7
11,Monitor,1100,4.6
6,External HD,400,4.6



🔥 Spark .orderBy():
+-----------+-----+------+
|    product|price|rating|
+-----------+-----+------+
|   Notebook| 3500|   4.8|
|   Notebook| 3200|   4.9|
|    Monitor| 1200|   4.7|
|    Monitor| 1100|   4.6|
|External HD|  400|   4.6|
+-----------+-----+------+
only showing top 5 rows


---
## 5. Transformations

| Operation | Pandas | Spark |
|-----------|--------|-------|
| Add/modify column | `df["new"] = expr` | `df.withColumn("new", expr)` |
| Drop column | `df.drop(columns=["c"])` | `df.drop("c")` |
| Rename column | `df.rename(columns={"a": "b"})` | `df.withColumnRenamed("a", "b")` |
| Conditional value | `np.where(cond, a, b)` | `F.when(cond, a).otherwise(b)` |
| Fill nulls | `df["c"].fillna(v)` | `df.fillna({"c": v})` |
| Drop null rows | `df.dropna()` | `df.dropna()` |
| Custom function | `df["c"].apply(fn)` | `F.udf(fn)(col)` |

In [ ]:
# Creating new columns
pdf2 = pdf.copy()
pdf2["total"]        = pdf2["price"] * pdf2["quantity"]
pdf2["discounted"]   = (pdf2["price"] * 0.90).round(2)
pdf2["is_expensive"] = pdf2["price"] > 500

print("🐼 Pandas — new columns:")
display(pdf2[["product", "price", "quantity", "total", "discounted", "is_expensive"]].head(5))

sdf2 = (
    sdf
    .withColumn("total",        F.col("price") * F.col("quantity"))
    .withColumn("discounted",   F.round(F.col("price") * 0.90, 2))
    .withColumn("is_expensive", F.col("price") > 500)
)
print("\n🔥 Spark — withColumn:")
sdf2.select("product", "price", "quantity", "total", "discounted", "is_expensive").show(5)

🐼 Pandas — new columns:


,product,price,quantity,total,discounted,is_expensive
0,Notebook,3500,1,3500,3150.0,True
1,Mouse,85,3,255,76.5,False
2,Keyboard,150,2,300,135.0,False
3,Monitor,1200,1,1200,1080.0,True
4,Headset,250,2,500,225.0,False



🔥 Spark — withColumn:
+--------+-----+--------+-----+----------+------------+
| product|price|quantity|total|discounted|is_expensive|
+--------+-----+--------+-----+----------+------------+
|Notebook| 3500|       1| 3500|    3150.0|        true|
|   Mouse|   85|       3|  255|      76.5|       false|
|Keyboard|  150|       2|  300|     135.0|       false|
| Monitor| 1200|       1| 1200|    1080.0|        true|
| Headset|  250|       2|  500|     225.0|       false|
+--------+-----+--------+-----+----------+------------+
only showing top 5 rows


In [ ]:
# Conditional column
# Pandas: np.select()    |    Spark: F.when().when().otherwise()

pdf2["price_range"] = np.select(
    [pdf2["price"] < 100, pdf2["price"] < 500],
    ["Budget",            "Mid-range"],
    default="Premium"
)
print("🐼 Pandas np.select():")
display(pdf2[["product", "price", "price_range"]].drop_duplicates(subset=["price_range"]).sort_values("price"))

sdf2 = sdf2.withColumn(
    "price_range",
    F.when(F.col("price") < 100, "Budget")
     .when(F.col("price") < 500, "Mid-range")
     .otherwise("Premium")
)
print("\n🔥 Spark F.when().otherwise():")
sdf2.select("product", "price", "price_range").show()

🐼 Pandas np.select():


,product,price,price_range
1,Mouse,85,Budget
2,Keyboard,150,Mid-range
0,Notebook,3500,Premium



🔥 Spark F.when().otherwise():
+-----------+-----+-----------+
|    product|price|price_range|
+-----------+-----+-----------+
|   Notebook| 3500|    Premium|
|      Mouse|   85|     Budget|
|   Keyboard|  150|  Mid-range|
|    Monitor| 1200|    Premium|
|    Headset|  250|  Mid-range|
|     Webcam|  320|  Mid-range|
|External HD|  400|  Mid-range|
|   Pendrive|   60|     Budget|
|  USB Cable|   25|     Budget|
|    USB Hub|  120|  Mid-range|
|   Notebook| 3200|    Premium|
|    Monitor| 1100|    Premium|
|   Keyboard|  145|  Mid-range|
|      Mouse|   90|     Budget|
|    Headset|  280|  Mid-range|
+-----------+-----+-----------+



In [ ]:
# String operations
# Pandas uses the .str accessor; Spark has F.upper(), F.length(), F.concat(), etc.

pdf2["product_upper"] = pdf2["product"].str.upper()
pdf2["product_len"]   = pdf2["product"].str.len()
pdf2["label"]         = pdf2["product"] + " — " + pdf2["category"]

print("🐼 Pandas string ops:")
display(pdf2[["product", "product_upper", "product_len", "label"]].head(5))

sdf2 = (
    sdf2
    .withColumn("product_upper", F.upper(F.col("product")))
    .withColumn("product_len",   F.length(F.col("product")))
    .withColumn("label",         F.concat(F.col("product"), F.lit(" — "), F.col("category")))
)
print("\n🔥 Spark string functions:")
sdf2.select("product", "product_upper", "product_len", "label").show(5, truncate=False)

🐼 Pandas string ops:


,product,product_upper,product_len,label
0,Notebook,NOTEBOOK,8,Notebook — Computers
1,Mouse,MOUSE,5,Mouse — Peripherals
2,Keyboard,KEYBOARD,8,Keyboard — Peripherals
3,Monitor,MONITOR,7,Monitor — Computers
4,Headset,HEADSET,7,Headset — Audio



🔥 Spark string functions:
+--------+-------------+-----------+----------------------+
|product |product_upper|product_len|label                 |
+--------+-------------+-----------+----------------------+
|Notebook|NOTEBOOK     |8          |Notebook — Computers  |
|Mouse   |MOUSE        |5          |Mouse — Peripherals   |
|Keyboard|KEYBOARD     |8          |Keyboard — Peripherals|
|Monitor |MONITOR      |7          |Monitor — Computers   |
|Headset |HEADSET      |7          |Headset — Audio       |
+--------+-------------+-----------+----------------------+
only showing top 5 rows


In [ ]:
# Date operations
pdf2["sale_date"]  = pd.to_datetime(pdf2["sale_date"])
pdf2["year"]       = pdf2["sale_date"].dt.year
pdf2["month"]      = pdf2["sale_date"].dt.month
pdf2["month_name"] = pdf2["sale_date"].dt.strftime("%B")

print("🐼 Pandas date ops:")
display(pdf2[["sale_date", "year", "month", "month_name"]].head(5))

sdf2 = (
    sdf2
    .withColumn("sale_date",  F.to_date(F.col("sale_date"), "yyyy-MM-dd"))
    .withColumn("year",       F.year(F.col("sale_date")))
    .withColumn("month",      F.month(F.col("sale_date")))
    .withColumn("month_name", F.date_format(F.col("sale_date"), "MMMM"))
)
print("\n🔥 Spark date functions:")
sdf2.select("sale_date", "year", "month", "month_name").show(5)

🐼 Pandas date ops:


,sale_date,year,month,month_name
0,2024-01-15,2024,1,January
1,2024-01-18,2024,1,January
2,2024-02-03,2024,2,February
3,2024-02-10,2024,2,February
4,2024-02-14,2024,2,February



🔥 Spark date functions:
+----------+----+-----+----------+
| sale_date|year|month|month_name|
+----------+----+-----+----------+
|2024-01-15|2024|    1|   January|
|2024-01-18|2024|    1|   January|
|2024-02-03|2024|    2|  February|
|2024-02-10|2024|    2|  February|
|2024-02-14|2024|    2|  February|
+----------+----+-----+----------+
only showing top 5 rows


In [ ]:
# Handling nulls
avg_pd = pdf["rating"].mean()
pdf2["rating"] = pdf2["rating"].fillna(avg_pd)
print(f"🐼 Pandas — filled nulls with mean: {avg_pd:.3f}")
print(f"   Nulls remaining: {pdf2['rating'].isnull().sum()}")

avg_sp = sdf.select(F.avg("rating")).collect()[0][0]
sdf2 = sdf2.fillna({"rating": round(avg_sp, 3)})
print(f"\n🔥 Spark — filled nulls with mean: {avg_sp:.3f}")
print(f"   Nulls remaining: {sdf2.filter(F.col('rating').isNull()).count()}")

🐼 Pandas — filled nulls with mean: 4.364
   Nulls remaining: 0

🔥 Spark — filled nulls with mean: nan
   Nulls remaining: 0


### 5.1 UDFs — User Defined Functions

> **UDFs** let you use custom Python functions inside Spark transformations.  
> They are the Spark equivalent of Pandas `.apply()`.  

> UDFs are slower than native `F.*` functions because they break out of the JVM.  
> Always prefer built-in Spark functions when they exist.

In [ ]:
def classify_price(price):
    """Classifies a price as Budget / Mid-range / Premium."""
    if price is None:  return "Unknown"
    if price < 100:    return "Budget"
    if price < 500:    return "Mid-range"
    return "Premium"

# 🐼 Pandas — .apply()
pdf2["price_class"] = pdf2["price"].apply(classify_price)
print("🐼 Pandas .apply():")
display(pdf2[["product", "price", "price_class"]].head(5))

# 🔥 Spark — UDF
classify_udf = F.udf(classify_price, StringType())
sdf2 = sdf2.withColumn("price_class", classify_udf(F.col("price")))
print("\n🔥 Spark UDF:")
sdf2.select("product", "price", "price_class").show(5)

🐼 Pandas .apply():


,product,price,price_class
0,Notebook,3500,Premium
1,Mouse,85,Budget
2,Keyboard,150,Mid-range
3,Monitor,1200,Premium
4,Headset,250,Mid-range



🔥 Spark UDF:
+--------+-----+-----------+
| product|price|price_class|
+--------+-----+-----------+
|Notebook| 3500|    Premium|
|   Mouse|   85|     Budget|
|Keyboard|  150|  Mid-range|
| Monitor| 1200|    Premium|
| Headset|  250|  Mid-range|
+--------+-----+-----------+
only showing top 5 rows


In [ ]:
# Renaming and dropping columns
pdf3 = pdf2.rename(columns={"price": "price_usd", "rating": "score"})
pdf3 = pdf3.drop(columns=["product_upper", "product_len", "label"])
print("🐼 Pandas columns after rename/drop:")
print(pdf3.columns.tolist())

sdf3 = (
    sdf2
    .withColumnRenamed("price", "price_usd")
    .withColumnRenamed("rating", "score")
    .drop("product_upper", "product_len", "label")
)
print("\n🔥 Spark columns after rename/drop:")
print(sdf3.columns)

🐼 Pandas columns after rename/drop:
['order_id', 'product', 'category', 'price_usd', 'quantity', 'score', 'region', 'sale_date', 'total', 'discounted', 'is_expensive', 'price_range', 'year', 'month', 'month_name', 'price_class']

🔥 Spark columns after rename/drop:
['order_id', 'product', 'category', 'price_usd', 'quantity', 'score', 'region', 'sale_date', 'total', 'discounted', 'is_expensive', 'price_range', 'year', 'month', 'month_name', 'price_class']


In [ ]:
spark.stop()